In [ ]:
#### region bedsets were created using "/sharehome/dlafonta/manuscripts/RNA/Figure_zoo/munge_cCRE_annotations.ipynb"

In [3]:
import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none'

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],  
    'font.size': 9,           
    'axes.labelsize': 9,    
    'xtick.labelsize': 7,     
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

# --- config ---
BW = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_mega120mPD_DpnII_R12_20250129_5kb.bw"


#### bedsets were created using "/sharehome/dlafonta/manuscripts/RNA/Figure_zoo/munge_cCRE_annotations.ipynb"
bed_sets = {   # label -> path
    "pELS": "/abyss/dlafonta/deeptools/regions/cCREs/20251110_pELS_cCRE_annot.bed",
    "PLS":  "/abyss/dlafonta/deeptools/regions/cCREs/20251110_PLS_cCRE_annot.bed",
    "CA-H3K4me3": "/abyss/dlafonta/deeptools/regions/cCREs/20251110_CAH3K4me3_cCRE_annot.bed",
    "dELS": "/abyss/dlafonta/deeptools/regions/cCREs/20251110_dELS_cCRE_annot.bed",
        "CA-only": "/abyss/dlafonta/deeptools/regions/cCREs/20251110_CAonly_cCRE_annot.bed",
    "CA-CTCF": "/abyss/dlafonta/deeptools/regions/cCREs/20251110_CACTCF_cCRE_annot.bed",
    "CA-TF": "/abyss/dlafonta/deeptools/regions/cCREs/20251110_CATF_cCRE_annot.bed",
    
}
colors = {
    "PLS":        "#FF0000",   # red        – promoter-like
    "pELS":       "#FFA700",   # orange     – proximal enhancer-like
    "dELS":       "#FFCD00",   # yellow     – distal enhancer-like
    "CA-H3K4me3": "#FFAAAA",   # pink/salmon – chromatin-accessible + H3K4me3
    "CA-CTCF":    "#00B0F0",   # cyan/blue  – chromatin-accessible + CTCF
    "CA-TF":      "#BE28E5",   # purple     – chromatin-accessible + TF
    "CA-only":    "#06DA93",   # green      – chromatin-accessible only
}

flank = 10000
nbins = 100
half  = 5                       # center bins for sorting

edges   = np.linspace(-flank, flank, nbins + 1)
centers = (edges[:-1] + edges[1:]) / 2
c0, c1  = nbins // 2 - half, nbins // 2 + half

# --- build a stackup per BED ---
stacks   = {}
profiles = {}
with bbi.open(BW) as f:
    bw_chroms = set(f.chromsizes)
    
    for label, path in bed_sets.items():
        reg = pd.read_csv(path, sep="\t", header=None,
                          names=["chrom", "start", "end", "ccre_id", "ccre_class"])

        # keep only chroms present in the bigWig
        reg = reg[reg.chrom.isin(bw_chroms)].reset_index(drop=True)

        mid    = (reg.start + reg.end) // 2
        starts = (mid - flank).values
        ends   = (mid + flank).values

        s = f.stackup(reg.chrom.values, starts, ends,
                      bins=nbins, missing=np.nan, oob=np.nan)

        # drop empty rows
        s = s[~np.isnan(s).all(axis=1)]

        # sort by center signal (guard NaN-center rows to bottom)
        key   = np.nanmean(s[:, c0:c1], axis=1)
        key   = np.where(np.isnan(key), -np.inf, key)
        s     = s[np.argsort(key)[::-1]]

        stacks[label]   = s
        profiles[label] = np.nanmean(s, axis=0)

# --- shared color scale across all heatmaps (robust percentiles) ---
allvals = np.concatenate([s[np.isfinite(s)].ravel() for s in stacks.values()])
vmin, vmax = np.nanpercentile(allvals, [2, 98])

# --- figure: 1 profile row + 1 heatmap row, one column per BED ---
labels = list(stacks.keys())
n = len(labels)

row_counts = np.array([len(stacks[l]) for l in labels], dtype=float)

# relative panel heights (proportional), with a floor so tiny classes stay visible
rel = row_counts / row_counts.max()
rel = np.maximum(rel, 0.15)              # smallest panel ≥ 15% of tallest
prof_rel = 1.33                           # profile panel height (relative units)
heights = [prof_rel] + list(rel)

# FIXED total figure height — does NOT scale with region count
fig_h = 2 + 2.2 * n                      # ~2.2 in per class; tune as needed
fig, axes = plt.subplots(
    n + 1, 1,
    figsize=(4, fig_h),
    height_ratios=heights,
    gridspec_kw={"hspace": 0.18},
    sharex=True,
)

# --- top: overlaid profiles ---
ax_prof = axes[0]
for label in labels:
    ax_prof.plot(centers, profiles[label],
                 color=colors[label], label=label, lw=1)
#ax_prof.axhline(0, color="grey", lw=0.6, ls="--")
ax_prof.set_ylabel("Mean LOS\nresidual")
ax_prof.margins(x=0)
#ax_prof.legend(fontsize=7, frameon=False, ncol=2, loc="upper right")

# --- stacked heatmaps ---
im = None
for i, label in enumerate(labels):
    ax = axes[i + 1]
    im = ax.imshow(
        stacks[label], aspect="auto", cmap="magma",
        vmin=vmin, vmax=vmax, interpolation="none",
        extent=[edges[0], edges[-1], 0, len(stacks[label])],
    )
    ax.set_ylabel(f"{label}\n(n={len(stacks[label])})",
                  rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])
    if i < n - 1:
        ax.tick_params(labelbottom=False)

    # colored outline matching the class
    for spine in ax.spines.values():
        spine.set_edgecolor(colors[label])
        spine.set_linewidth(2)
        spine.set_visible(True)

axes[-1].set_xlabel("Distance from cCRE center (bp)")
for spine in ax.spines.values():
    spine.set_linewidth(1.3)
ax.tick_params(width=1.3, length=3)
    
# --- shared colorbar ---
fig.subplots_adjust(right=0.86)
cax = fig.add_axes([0.88, 0.15, 0.05, 0.6])
fig.colorbar(im, cax=cax, label="LOS residual")

#plt.savefig("cCRE_class_stackups_vertical.pdf", bbox_inches="tight")
plt.savefig("cCRE_class_stackups_vertical.svg", bbox_inches="tight")
#plt.savefig("cCRE_class_stackups_vertical.png", dpi=200, bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_1366254/3523769488.py:78: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3523769488.py:78: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3523769488.py:78: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3523769488.py:78: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3523769488.py:78: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3523769488.py:78: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3523769488.py:78: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none o

In [12]:
#####Extended Data Fig. 7b - cCRE stackup (Speckles only)


import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none'  # SVG: leave text as text

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],   # falls back if Arial not installed
    'font.size': 9,                 # base size for everything
    'axes.labelsize': 9,           # x/y axis labels
    'xtick.labelsize': 7,           # tick numbers/labels
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

# --- config ---
BW = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_mega120mPD_DpnII_R12_20250129_5kb.bw"


#### bedsets were created using "/sharehome/dlafonta/manuscripts/RNA/Figure_zoo/munge_cCRE_annotations.ipynb"
bed_sets = {   # label -> path
    "pELS": "/abyss/dlafonta/deeptools/regions/cCREs/20251120_pELS_Speckle_cCRE_annot.bed",
    "PLS":  "/abyss/dlafonta/deeptools/regions/cCREs/20251120_PLS_Speckle_cCRE_annot.bed",
    "CA-H3K4me3": "/abyss/dlafonta/deeptools/regions/cCREs/20251120_CA_H3K4me3_Speckle_cCRE_annot.bed",
    "dELS": "/abyss/dlafonta/deeptools/regions/cCREs/20251120_dELS_Speckle_cCRE_annot.bed",
    "CA-only": "/abyss/dlafonta/deeptools/regions/cCREs/20251120_CA_only_Speckle_cCRE_annot.bed",
    "CA-CTCF": "/abyss/dlafonta/deeptools/regions/cCREs/20251120_CA_CTCF_Speckle_cCRE_annot.bed",
    "CA-TF": "/abyss/dlafonta/deeptools/regions/cCREs/20251120_CA_TF_Speckle_cCRE_annot.bed",
    
}
colors = {
    "PLS":        "#FF0000",   # red        – promoter-like
    "pELS":       "#FFA700",   # orange     – proximal enhancer-like
    "dELS":       "#FFCD00",   # yellow     – distal enhancer-like
    "CA-H3K4me3": "#FFAAAA",   # pink/salmon – chromatin-accessible + H3K4me3
    "CA-CTCF":    "#00B0F0",   # cyan/blue  – chromatin-accessible + CTCF
    "CA-TF":      "#BE28E5",   # purple     – chromatin-accessible + TF
    "CA-only":    "#06DA93",   # green      – chromatin-accessible only
}

flank = 10000
nbins = 100
half  = 5                       # center bins for sorting

edges   = np.linspace(-flank, flank, nbins + 1)
centers = (edges[:-1] + edges[1:]) / 2
c0, c1  = nbins // 2 - half, nbins // 2 + half

# --- build a stackup per BED ---
stacks   = {}
profiles = {}
with bbi.open(BW) as f:
    bw_chroms = set(f.chromsizes)
    
    for label, path in bed_sets.items():
        reg = pd.read_csv(path, sep="\t", header=None,
                          names=["chrom", "start", "end", "ccre_id", "ccre_class"])

        # keep only chroms present in the bigWig
        reg = reg[reg.chrom.isin(bw_chroms)].reset_index(drop=True)

        mid    = (reg.start + reg.end) // 2
        starts = (mid - flank).values
        ends   = (mid + flank).values

        s = f.stackup(reg.chrom.values, starts, ends,
                      bins=nbins, missing=np.nan, oob=np.nan)

        # drop empty rows
        s = s[~np.isnan(s).all(axis=1)]

        # sort by center signal (guard NaN-center rows to bottom)
        key   = np.nanmean(s[:, c0:c1], axis=1)
        key   = np.where(np.isnan(key), -np.inf, key)
        s     = s[np.argsort(key)[::-1]]

        stacks[label]   = s
        profiles[label] = np.nanmean(s, axis=0)

# --- shared color scale across all heatmaps (robust percentiles) ---
allvals = np.concatenate([s[np.isfinite(s)].ravel() for s in stacks.values()])
vmin, vmax = np.nanpercentile(allvals, [2, 98])

# --- figure: 1 profile row + 1 heatmap row, one column per BED ---
labels = list(stacks.keys())
n = len(labels)

row_counts = np.array([len(stacks[l]) for l in labels], dtype=float)

# relative panel heights (proportional), with a floor so tiny classes stay visible
rel = row_counts / row_counts.max()
rel = np.maximum(rel, 0.15)              # smallest panel ≥ 15% of tallest
prof_rel = 0.98                           # profile panel height (relative units)
heights = [prof_rel] + list(rel)

# FIXED total figure height — does NOT scale with region count
fig_h = 2 + 2.2 * n                      # ~2.2 in per class; tune as needed
fig, axes = plt.subplots(
    n + 1, 1,
    figsize=(4, fig_h),
    height_ratios=heights,
    gridspec_kw={"hspace": 0.18},
    sharex=True,
)

# --- top: overlaid profiles ---
ax_prof = axes[0]
for label in labels:
    ax_prof.plot(centers, profiles[label],
                 color=colors[label], label=label, lw=1)
#ax_prof.axhline(0, color="grey", lw=0.6, ls="--")
ax_prof.set_ylabel("Mean LOS\nresidual")
ax_prof.margins(x=0)
#ax_prof.legend(fontsize=7, frameon=False, ncol=2, loc="upper right")

# --- stacked heatmaps ---
im = None
for i, label in enumerate(labels):
    ax = axes[i + 1]
    im = ax.imshow(
        stacks[label], aspect="auto", cmap="magma",
        vmin=vmin, vmax=vmax, interpolation="none",
        extent=[edges[0], edges[-1], 0, len(stacks[label])],
    )
    ax.set_ylabel(f"{label}\n(n={len(stacks[label])})",
                  rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])
    if i < n - 1:
        ax.tick_params(labelbottom=False)

    # colored outline matching the class
    for spine in ax.spines.values():
        spine.set_edgecolor(colors[label])
        spine.set_linewidth(2)
        spine.set_visible(True)

axes[-1].set_xlabel("Distance from cCRE center (bp)")
for spine in ax.spines.values():
    spine.set_linewidth(1.3)
ax.tick_params(width=1.3, length=3)
    
# --- shared colorbar ---
fig.subplots_adjust(right=0.86)
cax = fig.add_axes([0.88, 0.15, 0.05, 0.6])
fig.colorbar(im, cax=cax, label="LOS residual")

#plt.savefig("cCRE_class_stackups_vertical.pdf", bbox_inches="tight")
plt.savefig("cCRE_class_stackups_vertical_speckleONLY.svg", bbox_inches="tight")
#plt.savefig("cCRE_class_stackups_vertical.png", dpi=200, bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_1366254/3960061055.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3960061055.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3960061055.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3960061055.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3960061055.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/3960061055.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the followin

In [9]:
#####Extended Data Fig. 7c - cCRE stackup (Act123 only)


import bbi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42 
matplotlib.rcParams['svg.fonttype'] = 'none'  # SVG: leave text as text

matplotlib.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],   # falls back if Arial not installed
    'font.size': 9,                 # base size for everything
    'axes.labelsize': 9,           # x/y axis labels
    'xtick.labelsize': 7,           # tick numbers/labels
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'axes.titlesize': 9,
})

# --- config ---
BW = "/abyss/dlafonta/deeptools/LOS/LOS_residuals_mod/5kb/LOS_residuals_range2Mb_K562_mega120mPD_DpnII_R12_20250129_5kb.bw"


#### bedsets were created using "/sharehome/dlafonta/manuscripts/RNA/Figure_zoo/munge_cCRE_annotations.ipynb"
bed_sets = {   # label -> path
    "pELS": "/abyss/dlafonta/deeptools/regions/cCREs/20260311_pELS_Act123_cCRE_annot.bed",
    "PLS":  "/abyss/dlafonta/deeptools/regions/cCREs/20260311_PLS_Act123_cCRE_annot.bed",
    "CA-H3K4me3": "/abyss/dlafonta/deeptools/regions/cCREs/20260311_CA_H3K4me3_Act123_cCRE_annot.bed",
    "dELS": "/abyss/dlafonta/deeptools/regions/cCREs/20260311_dELS_Act123_cCRE_annot.bed",
    "CA-only": "/abyss/dlafonta/deeptools/regions/cCREs/20260311_CA_only_Act123_cCRE_annot.bed",
    "CA-CTCF": "/abyss/dlafonta/deeptools/regions/cCREs/20260311_CA_CTCF_Act123_cCRE_annot.bed",
    "CA-TF": "/abyss/dlafonta/deeptools/regions/cCREs/20260311_CA_TF_Act123_cCRE_annot.bed",
    
}
colors = {
    "PLS":        "#FF0000",   # red        – promoter-like
    "pELS":       "#FFA700",   # orange     – proximal enhancer-like
    "dELS":       "#FFCD00",   # yellow     – distal enhancer-like
    "CA-H3K4me3": "#FFAAAA",   # pink/salmon – chromatin-accessible + H3K4me3
    "CA-CTCF":    "#00B0F0",   # cyan/blue  – chromatin-accessible + CTCF
    "CA-TF":      "#BE28E5",   # purple     – chromatin-accessible + TF
    "CA-only":    "#06DA93",   # green      – chromatin-accessible only
}

flank = 10000
nbins = 100
half  = 5                       # center bins for sorting

edges   = np.linspace(-flank, flank, nbins + 1)
centers = (edges[:-1] + edges[1:]) / 2
c0, c1  = nbins // 2 - half, nbins // 2 + half

# --- build a stackup per BED ---
stacks   = {}
profiles = {}
with bbi.open(BW) as f:
    bw_chroms = set(f.chromsizes)
    
    for label, path in bed_sets.items():
        reg = pd.read_csv(path, sep="\t", header=None,
                          names=["chrom", "start", "end", "ccre_id", "ccre_class"])

        # keep only chroms present in the bigWig
        reg = reg[reg.chrom.isin(bw_chroms)].reset_index(drop=True)

        mid    = (reg.start + reg.end) // 2
        starts = (mid - flank).values
        ends   = (mid + flank).values

        s = f.stackup(reg.chrom.values, starts, ends,
                      bins=nbins, missing=np.nan, oob=np.nan)

        # drop empty rows
        s = s[~np.isnan(s).all(axis=1)]

        # sort by center signal (guard NaN-center rows to bottom)
        key   = np.nanmean(s[:, c0:c1], axis=1)
        key   = np.where(np.isnan(key), -np.inf, key)
        s     = s[np.argsort(key)[::-1]]

        stacks[label]   = s
        profiles[label] = np.nanmean(s, axis=0)

# --- shared color scale across all heatmaps (robust percentiles) ---
allvals = np.concatenate([s[np.isfinite(s)].ravel() for s in stacks.values()])
vmin, vmax = np.nanpercentile(allvals, [2, 98])

# --- figure: 1 profile row + 1 heatmap row, one column per BED ---
labels = list(stacks.keys())
n = len(labels)

row_counts = np.array([len(stacks[l]) for l in labels], dtype=float)

# relative panel heights (proportional), with a floor so tiny classes stay visible
rel = row_counts / row_counts.max()
rel = np.maximum(rel, 0.15)              # smallest panel ≥ 15% of tallest
prof_rel = 1.33                           # profile panel height (relative units)
heights = [prof_rel] + list(rel)

# FIXED total figure height — does NOT scale with region count
fig_h = 2 + 2.2 * n                      # ~2.2 in per class; tune as needed
fig, axes = plt.subplots(
    n + 1, 1,
    figsize=(4, fig_h),
    height_ratios=heights,
    gridspec_kw={"hspace": 0.18},
    sharex=True,
)

# --- top: overlaid profiles ---
ax_prof = axes[0]
for label in labels:
    ax_prof.plot(centers, profiles[label],
                 color=colors[label], label=label, lw=1)
#ax_prof.axhline(0, color="grey", lw=0.6, ls="--")
ax_prof.set_ylabel("Mean LOS\nresidual")
ax_prof.margins(x=0)
#ax_prof.legend(fontsize=7, frameon=False, ncol=2, loc="upper right")

# --- stacked heatmaps ---
im = None
for i, label in enumerate(labels):
    ax = axes[i + 1]
    im = ax.imshow(
        stacks[label], aspect="auto", cmap="magma",
        vmin=vmin, vmax=vmax, interpolation="none",
        extent=[edges[0], edges[-1], 0, len(stacks[label])],
    )
    ax.set_ylabel(f"{label}\n(n={len(stacks[label])})",
                  rotation=0, ha="right", va="center", fontsize=9)
    ax.set_yticks([])
    if i < n - 1:
        ax.tick_params(labelbottom=False)

    # colored outline matching the class
    for spine in ax.spines.values():
        spine.set_edgecolor(colors[label])
        spine.set_linewidth(2)
        spine.set_visible(True)

axes[-1].set_xlabel("Distance from cCRE center (bp)")
for spine in ax.spines.values():
    spine.set_linewidth(1.3)
ax.tick_params(width=1.3, length=3)
    
# --- shared colorbar ---
fig.subplots_adjust(right=0.86)
cax = fig.add_axes([0.88, 0.15, 0.05, 0.6])
fig.colorbar(im, cax=cax, label="LOS residual")

#plt.savefig("cCRE_class_stackups_vertical.pdf", bbox_inches="tight")
plt.savefig("cCRE_class_stackups_vertical_Act123ONLY.svg", bbox_inches="tight")
#plt.savefig("cCRE_class_stackups_vertical.png", dpi=200, bbox_inches="tight")
plt.close(fig)

/tmp/ipykernel_1366254/550005455.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/550005455.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/550005455.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/550005455.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/550005455.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/550005455.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
/tmp/ipykernel_1366254/550005455.py:81: RuntimeWarning: Mean of empty slice
  key   = np.nanmean(s[:, c0:c1], axis=1)
findfont: Generic family 'sans-serif' not found because none of the following families were found: Arial
findfont: Generic family 'sans-serif' not found because none of the f